# Exercise 3 - What counts as a "hit"? Compound detector thresholds

**Time budget:** ~35 min &nbsp;|&nbsp; **Reference:** notebooks 03 + 04 &nbsp;|&nbsp; **No HUXt runs**

The default label is a single rule: enhancement detector >= 0.25. A forecaster might instead care
only about arrivals that are *also fast* (geoeffective). Because `results.csv` stores
`max_speed_enhancement` **and** `peak_vsw` per sample, you can redefine "hit" and re-fit the
classifier with **no HUXt reruns**, then see how the decision map moves.

You are given a `fit_hit(label)` helper, a longitude-v `GRID`, and a `show_maps(...)` plotter.

**By the end you can:** treat the label definition as a knob and explain how it reshapes the surrogate.

In [ ]:
# --- Google Colab bootstrap (no-op locally) ---
import sys, os

if "google.colab" in sys.modules:
    REPO_URL = os.environ.get("CONECAST_REPO", "https://github.com/georgemilosh/conecast")
    REPO_DIR = "/content/conecast"
    if not os.path.isdir(REPO_DIR):
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    try:
        import sunpy, huxt, wsaplus  # noqa: F401
        print("Colab bootstrap complete; cwd =", os.getcwd())
    except ModuleNotFoundError:
        print("Installing sunpy + WSA+ + HUXt (one-time, ~2 min)...")
        os.system("pip install -q sunpy wsaplus")
        os.system("pip install -q "
                  "'huxt @ git+https://github.com/University-of-Reading-Space-Science/HUXt'")
        print("Done - restarting the runtime. When it reconnects, RUN THIS CELL AGAIN.")
        os.kill(os.getpid(), 9)
else:
    print("Not in Colab - using the local checkout.")

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
BASE_DIR = cwd if (cwd / "scripts").exists() else cwd.parent
SCRIPT_DIR = BASE_DIR / "scripts"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
print("BASE_DIR =", BASE_DIR)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import ConstantKernel, Matern
from sklearn.preprocessing import StandardScaler
import gp_huxt_surrogate as gp

PARAM_NAMES = gp.PARAM_NAMES
res = pd.read_csv(BASE_DIR / "runs" / "gp_surrogate" / "2017-09-06" / "results.csv")
res = res.loc[res["status"] == "completed"].copy()
res["hit_simple"] = res["max_speed_enhancement"] >= 0.25

ctx = gp.load_huxt_context("2017-09-06", BASE_DIR / "data_dir" / "sw"); theta0 = ctx["theta0"]
span = np.array([1.0, 30.0, 20.0, 40.0, 0.25 * theta0[4]]); low, high = theta0 - span, theta0 + span
x = res[PARAM_NAMES].to_numpy(float); xs = StandardScaler().fit(x)

def fit_hit(label):
    """Fit a GP hit-classifier on res[label]; return predict(theta_grid) -> P(hit)."""
    clf = GaussianProcessClassifier(
        ConstantKernel(1.0, (1e-3, 1e5)) * Matern(np.ones(5), (1e-2, 1e5), nu=2.5),
        random_state=42).fit(xs.transform(x), res[label].astype(int))
    return lambda grid: clf.predict_proba(xs.transform(grid))[:, 1]

xi, yi, n = 1, 4, 60                                   # longitude (1) vs v (4)
gx = np.linspace(low[xi], high[xi], n); gy = np.linspace(low[yi], high[yi], n)
XX, YY = np.meshgrid(gx, gy)
GRID = np.tile(theta0, (n * n, 1)); GRID[:, xi] = XX.ravel(); GRID[:, yi] = YY.ravel()

def show_maps(P_simple, P_compound):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for ax, (name, P) in zip(axes, [("hit_simple", P_simple), ("hit_compound", P_compound)]):
        Pg = np.asarray(P).reshape(n, n)
        c = ax.contourf(XX, YY, Pg, levels=20, cmap="viridis"); ax.contour(XX, YY, Pg, [0.5], colors="white")
        ax.plot(theta0[xi], theta0[yi], "w*", ms=12); ax.set_title(name); ax.set_xlabel("longitude")
    axes[0].set_ylabel("v"); fig.colorbar(c, ax=axes[1]); plt.tight_layout()

print("simple hit rate:", round(res["hit_simple"].mean(), 3),
      "| peak_vsw range:", int(res.peak_vsw.min()), "-", int(res.peak_vsw.max()))

## Your task

Define a **compound** hit (fast *and* enhanced), measure how the label changes, then compare the
two decision maps.

In [ ]:
# ------------------------------ YOUR CODE HERE ------------------------------

ENH_THR, VMIN = 0.25, 700.0     # tune VMIN later in the stretch goal
# 1. Build res["hit_compound"]: a hit needs max_speed_enhancement >= ENH_THR AND peak_vsw >= VMIN.
# 2. Print the compound hit rate, and how many samples flip from hit (simple) to miss (compound);
#    peek at the flipped rows' longitude / v / peak_vsw - are they slow, glancing, or both?
# 3. Fit a classifier for EACH label with fit_hit(...) and evaluate P(hit) on GRID
#    -> P_simple, P_compound.
# 4. show_maps(P_simple, P_compound).


## Questions

1. Which way does the `P(hit)=0.5` boundary move when you add the speed floor, and in which
   parameter (longitude? v?) is it most sensitive?
2. The flipped samples - slow CMEs, glancing ones, or both?
3. A compound label has fewer hits over a smaller region. Connect to Exercise 2: would this
   stricter label need *more* HUXt runs to pin down?

### Stretch
- Sweep `VMIN` over {600, 800, 1000} and watch the hit region contract.
- Add `& res["front_hit"]` (geometric arrival) as a third condition and compare.
- Repeat the local + global feature importance (notebook 04, Tasks 4b/4c) under the compound
  label - does the *ranking* of which parameter matters change?